In [ ]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()  # 加载.env文件里的变量

llm = ChatOpenAI(
    model="deepseek-chat",  # 使用的模型名称，目前官方推荐用 'deepseek-chat'
    api_key=os.getenv("DEEPSEEK_API_KEY"),  # 你的 DeepSeek API Key
    base_url="https://api.deepseek.com/v1",  # DeepSeek API 地址
    temperature=0,
)

In [ ]:

import operator
from typing import Annotated, List, TypedDict

from langgraph.graph import StateGraph,START,END
from langchain_core.messages import HumanMessage,SystemMessage


class State(TypedDict):
    messages: Annotated[List[str],operator.add]
    
builder=StateGraph(State)

def chat_with_model(state):
    print(state)
    print("-------")
    messages=state['messages']
    response=llm.invoke(messages)
    return {'messages':[response]}

def convert_messages(state):
    prompt="""
    你是一位数据提取专家，负责从文本中检索关键信息，请为所提供的文本提取相关信息，并以json格式输出，概述锁提取的关键数据点。
    """
    
    print(state)
    print("----------------")
    messages=state['messages']
    messages=messages[-1]
    
    messages=[
        SystemMessage(content=prompt),
        HumanMessage(messages.content)
    ]
    
    response=llm.invoke(messages)
    return {'messages':[response]}

builder.add_node('chat_with_model',chat_with_model)
builder.add_node('convert_messages',convert_messages)

builder.add_edge(START,'chat_with_model')
builder.add_edge('chat_with_model','convert_messages')
builder.add_edge('convert_messages',END)

graph=builder.compile()

In [ ]:
from IPython.display import display,Image

display(Image(graph.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
initial_state={'messages':['请详细的介绍一下你自己']}
result=graph.invoke(initial_state)
print(result['messages'][-1].content)